# Etapa 5 — Preparação para Deploy no Hugging Face Spaces

Este notebook prepara e valida todos os artefatos necessários para o deploy.

**Arquivos que precisam ir para o Space:**
- `app.py` — interface Streamlit
- `requirements.txt` — dependências
- `README.md` — metadados do Space
- `modelo_final.pkl` — modelo LightGBM treinado
- `pipeline_preprocessamento.pkl` — pipeline de transformação
- `features.pkl` — lista de features na ordem correta
- `class_weight.pkl` — pesos das classes
- `metadados_modelo.json` — métricas e threshold

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Verificação dos Artefatos

In [ ]:
import os, json, joblib
import numpy as np

# Artefatos obrigatórios para o deploy
ARTEFATOS = [
    '/content/drive/MyDrive/artefatos/modelo_final.pkl',
    '/content/drive/MyDrive/artefatos/pipeline_preprocessamento.pkl',
    '/content/drive/MyDrive/artefatos/features.pkl',
    '/content/drive/MyDrive/artefatos/class_weight.pkl',
    '/content/drive/MyDrive/artefatos/metadados_modelo.json',
]

print('Verificando artefatos...')
todos_ok = True
for path in ARTEFATOS:
    existe = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if existe else 0
    status = '✅' if existe else '❌'
    print(f'  {status} {path:<45} {size:>8.1f} KB')
    if not existe:
        todos_ok = False

print(f'\n{"✅ Todos os artefatos encontrados!" if todos_ok else "❌ Artefatos faltando!"}')

Verificando artefatos...
  ✅ /content/drive/MyDrive/artefatos/modelo_final.pkl    171.6 KB
  ✅ /content/drive/MyDrive/artefatos/pipeline_preprocessamento.pkl      4.7 KB
  ✅ /content/drive/MyDrive/artefatos/features.pkl      0.5 KB
  ✅ /content/drive/MyDrive/artefatos/class_weight.pkl      0.1 KB
  ✅ /content/drive/MyDrive/artefatos/metadados_modelo.json      0.5 KB

✅ Todos os artefatos encontrados!


## Teste de Predição Antes do Deploy

Valida que o pipeline completo funciona corretamente antes de subir para o HF.

In [ ]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

# Redefine OutlierClipper antes de carregar pipeline
class OutlierClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.lower_ = X_df.quantile(self.lower)
        self.upper_ = X_df.quantile(self.upper)
        return self
    def transform(self, X, y=None):
        return pd.DataFrame(X).copy().clip(
            lower=self.lower_, upper=self.upper_, axis=1
        ).values

# Carrega artefatos
pipeline = joblib.load('/content/drive/MyDrive/artefatos/pipeline_preprocessamento.pkl')
modelo   = joblib.load('/content/drive/MyDrive/artefatos/modelo_final.pkl')
features = joblib.load('/content/drive/MyDrive/artefatos/features.pkl')

with open('/content/drive/MyDrive/artefatos/metadados_modelo.json') as f:
    meta = json.load(f)

THRESHOLD = meta['threshold_otimo']
print(f'✅ Artefatos carregados | Threshold: {THRESHOLD}')

✅ Artefatos carregados | Threshold: 0.473


In [ ]:
def criar_features(dados):
    d = pd.DataFrame([dados])
    d['imc_categoria'] = pd.cut(
        d['imc'], bins=[0,18.5,25,30,35,100],
        labels=[0,1,2,3,4], right=False
    ).astype(float)
    d['n_fatores_risco']       = (d['pressao_alta'] + d['colesterol_alto'] +
                                  (d['imc']>=30).astype(float) +
                                  d['doenca_cardiaca'] + d['avc'])
    d['idade_x_saude_geral']   = d['faixa_etaria'] * d['saude_geral']
    d['score_saudavel']        = (d['atividade_fisica'] + d['consume_frutas'] +
                                  d['consume_vegetais'] + (1-d['fumante']) +
                                  (1-d['alcool_pesado']))
    d['score_socioeconomico']  = d['renda'] + d['escolaridade']
    d['alto_risco_combinado']  = ((d['faixa_etaria']>=9) & (d['saude_geral']>=4)).astype(float)
    return d[features]

# Paciente alto risco
pac_alto = {
    'pressao_alta':1,'colesterol_alto':1,'checou_colesterol':1,'imc':36.0,
    'fumante':1,'avc':0,'doenca_cardiaca':1,'atividade_fisica':0,
    'consume_frutas':0,'consume_vegetais':0,'alcool_pesado':0,'plano_saude':1,
    'sem_medico_por_custo':0,'saude_geral':4,'dias_saude_mental_ruim':5,
    'dias_saude_fisica_ruim':10,'dificuldade_caminhar':1,'sexo':0,
    'faixa_etaria':10,'escolaridade':4,'renda':3
}

# Paciente baixo risco
pac_baixo = {
    'pressao_alta':0,'colesterol_alto':0,'checou_colesterol':1,'imc':22.5,
    'fumante':0,'avc':0,'doenca_cardiaca':0,'atividade_fisica':1,
    'consume_frutas':1,'consume_vegetais':1,'alcool_pesado':0,'plano_saude':1,
    'sem_medico_por_custo':0,'saude_geral':1,'dias_saude_mental_ruim':0,
    'dias_saude_fisica_ruim':0,'dificuldade_caminhar':0,'sexo':1,
    'faixa_etaria':3,'escolaridade':6,'renda':8
}

for nome, pac in [('Alto Risco', pac_alto), ('Baixo Risco', pac_baixo)]:
    df_pac = criar_features(pac)
    X_pac  = pipeline.transform(df_pac)
    prob   = float(modelo.predict_proba(X_pac)[0, 1])
    diag   = 'COM DIABETES' if prob >= THRESHOLD else 'SEM DIABETES'
    print(f'  {nome:<12}: {diag} ({prob*100:.1f}%)')

print('\n✅ Pipeline de predição validado!')

  Alto Risco  : COM DIABETES (88.9%)
  Baixo Risco : SEM DIABETES (2.6%)

✅ Pipeline de predição validado!


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Copiar Artefatos para a Pasta do Space

Cria a pasta `hf_space/` com todos os arquivos prontos para upload no Hugging Face.

In [ ]:
import shutil

PASTA_SPACE = 'hf_space'
os.makedirs(PASTA_SPACE, exist_ok=True)

# Copia artefatos do modelo
for arq in [
    '/content/drive/MyDrive/artefatos/modelo_final.pkl',
    '/content/drive/MyDrive/artefatos/pipeline_preprocessamento.pkl',
    '/content/drive/MyDrive/artefatos/features.pkl',
    '/content/drive/MyDrive/artefatos/class_weight.pkl',
    '/content/drive/MyDrive/artefatos/metadados_modelo.json',
]:
    nome = os.path.basename(arq)
    shutil.copy(arq, f'{PASTA_SPACE}/{nome}')
    print(f'  ✅ {nome}')

print(f'\n📁 Conteúdo de {PASTA_SPACE}/:')
for f in sorted(os.listdir(PASTA_SPACE)):
    size = os.path.getsize(f'{PASTA_SPACE}/{f}') / 1024
    print(f'   ├── {f:<45} {size:>8.1f} KB')

print('\n⚠️  Ainda falta adicionar manualmente:')
print('   ├── app.py')
print('   ├── requirements.txt')
print('   └── README.md')

  ✅ modelo_final.pkl
  ✅ pipeline_preprocessamento.pkl
  ✅ features.pkl
  ✅ class_weight.pkl
  ✅ metadados_modelo.json

📁 Conteúdo de hf_space/:
   ├── class_weight.pkl                                   0.1 KB
   ├── features.pkl                                       0.5 KB
   ├── metadados_modelo.json                              0.5 KB
   ├── modelo_final.pkl                                 171.6 KB
   ├── pipeline_preprocessamento.pkl                      4.7 KB

⚠️  Ainda falta adicionar manualmente:
   ├── app.py
   ├── requirements.txt
   └── README.md
